# 08 — Sequences & LSTMs (PyTorch)

Static models (XGBoost, MLP) see time-columns as **unordered**. But a borrower's payment history is a
**sequence**, and its *order* (improving vs deteriorating) carries signal. A **recurrent** network
reads the sequence in order, carrying a **memory** (hidden state) forward. The **LSTM** is the robust
version, with gates that control what to remember/forget.

Data: Taiwan Credit Card, 30,000 accounts, 6 months of payment status per account.

> torch-only notebook (xgboost segfaults if imported alongside torch on this Mac). The XGBoost
> baseline below is measured in a separate process on the identical split.

## 1. What "memory" means — a tiny RNN by hand

A recurrent net keeps one number `h` (memory) and updates it each month:
`h = tanh(0.5*thisMonth + 0.7*previousMemory)`. Watch it separate two borrowers, and note that
**reordering the same values flips the answer** — order is the signal.

In [1]:
import numpy as np
months = ["Apr","May","Jun","Jul","Aug","Sep"]
def run_rnn(seq, wx=0.5, wh=0.7):
    h=0.0; tr=[]
    for x in seq: h=np.tanh(wx*x+wh*h); tr.append(h)
    return tr
improving=[2,-1,-1,-1,-1,-1]; deteriorating=[-2,-2,-1,-1,2,2]
for name,s in [("IMPROVING (safe)",improving),("DETERIORATING (defaulted)",deteriorating)]:
    print(name, "-> final memory", round(run_rnn(s)[-1],2))
print("reversed deteriorating (same values):", round(run_rnn(deteriorating[::-1])[-1],2), "-> flips to safe")

IMPROVING (safe) -> final memory -0.77
DETERIORATING (defaulted) -> final memory 0.86
reversed deteriorating (same values): -0.92 -> flips to safe


## 2. Shape the data into sequences

An LSTM wants input shaped `(batch, timesteps, features)`. We take the 6 payment-status columns in
chronological order and reshape to `(N, 6, 1)`.

In [2]:
import pandas as pd, torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
torch.manual_seed(42); np.random.seed(42)

df = pd.read_csv("../data/raw/taiwan_credit/UCI_Credit_Card.csv")
target = "default.payment.next.month"
seq_cols = ["PAY_6","PAY_5","PAY_4","PAY_3","PAY_2","PAY_0"]   # Apr -> Sep
y = df[target].values.astype("float32")
seq = df[seq_cols].values.astype("float32")
seq = (seq - seq.mean())/seq.std()
seq = seq[:, :, None]                      # (N, 6, 1)
print("sequence tensor:", seq.shape, "= (borrowers, months, features)")
Xtr,Xte,ytr,yte = train_test_split(seq,y,test_size=.2,stratify=y,random_state=42)
Xtr,Xva,ytr,yva = train_test_split(Xtr,ytr,test_size=.25,stratify=ytr,random_state=42)
tt=lambda a: torch.tensor(a)
Xtr_t,Xva_t,Xte_t=tt(Xtr),tt(Xva),tt(Xte); ytr_t=torch.tensor(ytr).unsqueeze(1)
spw=(ytr==0).sum()/(ytr==1).sum()

sequence tensor: (30000, 6, 1) = (borrowers, months, features)


## 3. Build and train the LSTM

`nn.LSTM` does the recurrent loop internally. We take the **final hidden state** (the memory after the
last month) and pass it through a linear layer to a default probability.

In [3]:
class SeqLSTM(nn.Module):
    def __init__(self, hidden=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):                 # x: (batch, 6, 1)
        out, (hn, cn) = self.lstm(x)      # hn: (1, batch, hidden) final memory
        return self.head(hn[-1])

model=SeqLSTM(); print("params:", sum(p.numel() for p in model.parameters()))
criterion=nn.BCEWithLogitsLoss(pos_weight=torch.tensor([spw]))
opt=torch.optim.Adam(model.parameters(), lr=5e-3)
loader=DataLoader(TensorDataset(Xtr_t,ytr_t), batch_size=256, shuffle=True)
best,bs=0,None
for epoch in range(1,26):
    model.train()
    for xb,yb in loader:
        opt.zero_grad(); loss=criterion(model(xb),yb); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        va=roc_auc_score(yva, torch.sigmoid(model(Xva_t)).numpy().ravel())
    if va>best: best,bs=va,{k:v.clone() for k,v in model.state_dict().items()}
    if epoch in (1,5,10,15,20,25): print(f"  epoch {epoch:2d}  val AUC {va:.4f}")
model.load_state_dict(bs); model.eval()
with torch.no_grad():
    lstm_auc=roc_auc_score(yte, torch.sigmoid(model(Xte_t)).numpy().ravel())
print(f"\nLSTM test AUC = {lstm_auc:.4f}")

params: 4513


  epoch  1  val AUC 0.7193


  epoch  5  val AUC 0.7420


  epoch 10  val AUC 0.7459


  epoch 15  val AUC 0.7416


  epoch 20  val AUC 0.7449


  epoch 25  val AUC 0.7437

LSTM test AUC = 0.7365


## 4. Honest comparison

```
XGBoost (6 PAY cols, static, separate process) : 0.7409
LSTM    (6-month sequence)                      : ~0.737
```

XGBoost **still wins slightly**, even on payment-history data. Why: 6 months is short, so most signal
is in the single most-recent-month column (which the tree exploits), and LSTMs are data-hungry. The
sequence advantage is real but small at this length. **The LSTM's payoff grows with longer, richer
sequences, and in the HYBRID model (XGBoost static + LSTM temporal embedding), the project's headline.**

## 5. Recap
- A sequence's **order** carries signal (improving vs deteriorating) that unordered columns lose.
- A **recurrent** net reads step by step with a **memory** (hidden state); the **LSTM** adds gates
  (forget/input/output) to remember long-range info robustly.
- Data shape for an LSTM: `(batch, timesteps, features)`; use the **final hidden state** to predict.
- Honest result: on 6 short months the LSTM ~ties XGBoost. It wins with **longer/richer sequences**
  and inside a **hybrid** with XGBoost. Next: build that hybrid.